# 📊 Customer Churn Analysis — Phase 2: Cleaning & Merging

---

## What this notebook does

We take the five raw datasets saved in Phase 1 and transform them into one clean, analysis-ready master table. Every cleaning decision is documented with a reason — not just *what* we did, but *why* we did it this way instead of another.

## Problems inherited from Phase 1

| Problem | Dataset | Details |
|---|---|---|
| Duplicate rows | `customers` | 50 injected + any natural duplicates |
| Missing values | `customers` | `tenure` (~5%), `monthly_charges` (~3%), `total_charges` (11 rows) |
| Inconsistent category casing | `customers` | `contract` trailing spaces, `internet_service` UPPERCASE variants |
| Mixed column naming (CamelCase) | `customers` | `SeniorCitizen`, `MonthlyCharges`, etc. |
| Wrong data type | `customers` | `total_charges` loaded as string instead of float |
| Multiple rows per customer | `usage`, `tickets`, `billing` | Must aggregate before merging |

## Cleaning Order (matters!)

```
1.  Load raw data fresh
2.  Standardise all customer_id columns to uppercase string  ← prevents join failures
3.  Remove duplicates from customers                         ← before any filling
4.  Fix column names to snake_case                           ← before referencing by name
5.  Fix data types                                           ← before numeric operations
6.  Fix inconsistent categories (EXCLUDING customer_id)      ← after dtype fixes
7.  Handle missing values                                    ← after types are fixed
8.  Validate customers is fully clean
9.  Aggregate usage / tickets / billing
10. Merge all tables
11. Fill behavioural NaNs
12. Final validation
13. Save to processed/
```

## Notebook Map
```
01_data_collection.ipynb
02_cleaning_merging.ipynb   ← YOU ARE HERE
03_eda_feature_engineering.ipynb
04_modelling.ipynb
```

---
## 1. Setup — Drive, Libraries & Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import re
import pandas as pd
import numpy as np

BASE_PATH      = "/content/drive/MyDrive/customer-churn-project"
RAW_PATH       = os.path.join(BASE_PATH, "data/raw/")
PROCESSED_PATH = os.path.join(BASE_PATH, "data/processed/")

os.makedirs(PROCESSED_PATH, exist_ok=True)

pd.set_option("display.max_columns",  None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:.2f}".format)

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"RAW_PATH       → {RAW_PATH}")
print(f"PROCESSED_PATH → {PROCESSED_PATH}")

---
## 2. Load Raw Data

**Rule:** Every notebook always starts by loading from disk — never carry variables forward from another notebook. This makes each notebook independently runnable and prevents hidden state bugs.

We use `parse_dates` on date columns so pandas stores them as proper `datetime64` objects instead of plain strings. This matters when we compute `days_since_last_login` in Section 9.

In [ ]:
customers = pd.read_csv(f"{RAW_PATH}customers.csv")
usage     = pd.read_csv(f"{RAW_PATH}usage.csv",   parse_dates=["login_date"])
tickets   = pd.read_csv(f"{RAW_PATH}tickets.csv", parse_dates=["ticket_date"])
billing   = pd.read_csv(f"{RAW_PATH}billing.csv", parse_dates=["billing_date"])

print("Shapes on load (rows × cols):")
for name, frame in [("customers", customers), ("usage", usage),
                    ("tickets",  tickets),    ("billing", billing)]:
    print(f"  {name:<12} {frame.shape[0]:>6,} × {frame.shape[1]}")

---
## 3. Standardise `customer_id` Across All Tables

**This step must happen before everything else — it is the most critical step in the entire notebook.**

The Telco customer IDs look like `7590-VHVEG`. When these are saved to CSV and reloaded, they are fine. But later in this notebook we apply `.str.title()` to fix inconsistent category casing in the `customers` table. `.str.title()` affects every string column it touches — including `customer_id`, converting `7590-VHVEG` → `7590-Vhveg`.

Meanwhile, the `usage`, `tickets`, and `billing` tables were simulated directly from the original uppercase IDs and never title-cased — they keep `7590-VHVEG`.

Result: `customers.customer_id` = `'7590-Vhveg'`, `usage.customer_id` = `'7590-VHVEG'`. Zero rows match on the join. Every behavioural column becomes NaN, then 0.

**The fix:** force all `customer_id` columns to the same canonical format — uppercase, stripped of whitespace — before any other transformation runs. We do this on all four tables at once so it can never be accidentally out-of-sync.

In [ ]:
for frame in [customers, usage, tickets, billing]:
    frame["customer_id"] = frame["customer_id"].astype(str).str.strip().str.upper()

# Verify all four tables now share the same ID format
print("customer_id samples after standardisation:")
print(f"  customers : {customers['customer_id'].head(3).tolist()}")
print(f"  usage     : {usage['customer_id'].head(3).tolist()}")
print(f"  tickets   : {tickets['customer_id'].head(3).tolist()}")
print(f"  billing   : {billing['customer_id'].head(3).tolist()}")

# Confirm IDs actually overlap — if any of these are 0 something is still wrong
print(f"\nID overlap checks:")
cust_ids = set(customers["customer_id"])
print(f"  customers ∩ usage  : {len(cust_ids & set(usage['customer_id'])):,}")
print(f"  customers ∩ tickets: {len(cust_ids & set(tickets['customer_id'])):,}")
print(f"  customers ∩ billing: {len(cust_ids & set(billing['customer_id'])):,}")

---
## 4. Remove Duplicates from `customers`

**Why first (after ID standardisation)?** Duplicates must be removed before filling missing values. If a row is duplicated and one copy has a null, filling first creates two slightly-different rows that are harder to detect as duplicates later.

**Why two steps?**
- Step 1: drop fully identical rows (our 50 injected duplicates)
- Step 2: check for same `customer_id` but different values — these indicate a data pipeline error and need explicit resolution

**Why NOT deduplicate `usage`, `tickets`, `billing`?** Those tables have one row per event. A customer can legitimately log in twice in one day or open two tickets in one month. We handle them through aggregation in Section 9.

In [ ]:
# Step 1 — drop fully identical rows
n_before     = len(customers)
customers    = customers.drop_duplicates()
n_full_dupes = n_before - len(customers)
print(f"Full duplicate rows removed: {n_full_dupes}")

# Step 2 — check for same customer_id with conflicting values
id_dupes = customers[customers.duplicated(subset="customer_id", keep=False)]
print(f"Remaining customer_id collisions: {id_dupes['customer_id'].nunique()}")

if len(id_dupes) > 0:
    print("\nSample of conflicting records:")
    print(id_dupes.head(4).to_string())
    # Keep the first occurrence — in a real project you would investigate which is correct
    customers = customers.drop_duplicates(subset="customer_id", keep="first")
    print("Kept first occurrence for each conflicting customer_id.")

print(f"\nCustomers shape after deduplication: {customers.shape}")

---
## 5. Standardise Column Names to snake_case

**Why snake_case?** The raw `customers` dataset uses inconsistent naming — `customer_id` (snake), `SeniorCitizen` (PascalCase), `MonthlyCharges` (camelCase). Mixed naming forces you to memorise capitalisation for every column and causes `KeyError` bugs.

**Why a regex approach instead of a manual rename dict?** The regex `([a-z])([A-Z])` finds every lowercase→uppercase boundary and inserts an underscore automatically. `SeniorCitizen` → `senior_citizen` without having to list it by hand.

**Why exclude `customer_id` from this step?** It is already correctly named and was standardised to uppercase values in Step 3. The regex only touches column *names*, not values, so it is safe here. But we explicitly verify it is unchanged.

In [ ]:
def to_snake_case(name):
    """
    Convert a column name to snake_case.
    'SeniorCitizen'  → 'senior_citizen'
    'MonthlyCharges' → 'monthly_charges'
    'customer_id'    → 'customer_id'  (unchanged)
    """
    name = re.sub(r"([a-z])([A-Z])", r"\1_\2", name)
    return name.strip().lower()

old_cols          = customers.columns.tolist()
customers.columns = [to_snake_case(c) for c in customers.columns]
new_cols          = customers.columns.tolist()

print("Column rename mapping:")
for old, new in zip(old_cols, new_cols):
    marker = " ← changed" if old != new else ""
    print(f"  {old:<22} → {new}{marker}")

# Safety check: customer_id must still be present and unchanged
assert "customer_id" in customers.columns, "ERROR: customer_id column lost during rename!"
print("\ncustomer_id column preserved ✓")

---
## 6. Fix Data Types

**Why before missing value handling?** If `total_charges` is stored as a string, any numeric operation on it (mean, median, fill) will crash. Types must be correct before we can do arithmetic.

**The `total_charges` problem:** The original Telco CSV stores `TotalCharges` as a string because some rows contain a space `" "` instead of a number (new customers with zero tenure). When pandas reads a space as a string the entire column becomes `object` dtype.

**Why `pd.to_numeric(errors='coerce')`?**
- `errors='raise'` crashes on the first bad value
- `errors='ignore'` silently leaves the column as strings
- `errors='coerce'` converts valid numbers and turns unparseable values into `NaN`, which we handle explicitly in the missing values step — the correct approach

In [ ]:
print("Dtypes before fix:")
print(customers[["tenure", "monthly_charges", "total_charges"]].dtypes)
print()

# Convert total_charges: string → float, spaces and blanks → NaN
customers["total_charges"] = pd.to_numeric(customers["total_charges"], errors="coerce")

# senior_citizen is 0/1 — ensure it is int not float (can become float after CSV roundtrip)
customers["senior_citizen"] = pd.to_numeric(customers["senior_citizen"], errors="coerce").fillna(0).astype(int)

print("Dtypes after fix:")
print(customers[["tenure", "monthly_charges", "total_charges"]].dtypes)
print()
print(f"total_charges nulls after coercion: {customers['total_charges'].isna().sum()}")
print("(These are new customers with 0 tenure — handled in Section 7)")

---
## 7. Fix Inconsistent Categories

**The problem:** `contract` has values like `"Month-to-month "` (trailing space) alongside `"Month-to-month"`. `internet_service` has `"FIBER OPTIC"` alongside `"Fiber optic"`. To pandas these are different strings.

**Why `.str.strip().str.title()`?**
- `.str.strip()` removes leading/trailing whitespace
- `.str.title()` normalises any casing variant to a single canonical form

**Critical: we must EXCLUDE `customer_id` from this operation.**
`customer_id` contains values like `7590-VHVEG`. Applying `.str.title()` would convert this to `7590-Vhveg`, which would no longer match the uppercase IDs in `usage`, `tickets`, and `billing`, causing every left join to produce zero matches and all behavioural columns to be zero. We explicitly exclude it from the loop.

In [ ]:
# Get all string columns EXCEPT customer_id
# customer_id must stay uppercase to match usage/tickets/billing on the join
str_cols = [
    c for c in customers.select_dtypes(include="object").columns
    if c != "customer_id"
]

for col in str_cols:
    customers[col] = customers[col].str.strip().str.title()

# Verify customer_id is still uppercase
print(f"customer_id sample after title-case step: {customers['customer_id'].head(3).tolist()}")
print("(Must still be UPPERCASE — e.g. '7590-VHVEG', not '7590-Vhveg')")
print()

# Verify category columns are now clean
print("contract (should be exactly 3 values):")
print(customers["contract"].value_counts().to_string())

print("\ninternet_service (should be exactly 3 values):")
print(customers["internet_service"].value_counts().to_string())

print("\npayment_method (should be exactly 4 values):")
print(customers["payment_method"].value_counts().to_string())

### 7b. Ensure `churn` is integer

After `.str.title()`, the `churn` column is unaffected if it was already numeric (0/1 int). But after a CSV round-trip it may have been stored as `'0'`/`'1'` strings — we handle both cases defensively.

In [ ]:
print(f"churn dtype:          {customers['churn'].dtype}")
print(f"churn unique values:  {sorted(customers['churn'].unique())}")

if customers["churn"].dtype == object:
    # Title-case turned '0'→'0', '1'→'1', 'yes'→'Yes', 'no'→'No'
    customers["churn"] = customers["churn"].map(
        {"Yes": 1, "No": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0}
    ).astype(int)
    print("Re-encoded churn to int.")
else:
    customers["churn"] = customers["churn"].astype(int)
    print("churn already numeric — cast to int confirmed.")

print(f"Churn rate: {customers['churn'].mean():.1%}")

---
## 8. Handle Missing Values

This is the most decision-heavy step. The wrong fill strategy can silently bias the model. We treat each column differently based on *why* it is missing.

### 8a. Missing value audit

In [ ]:
missing     = customers.isna().sum()
missing     = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(customers) * 100).round(2)

audit = pd.DataFrame({"missing_count": missing, "missing_%": missing_pct})
if len(audit) == 0:
    print("No missing values found — all clean already.")
else:
    print("Missing value audit:")
    print(audit.to_string())

### 8b. `tenure` — grouped median fill + missing flag

**Why not drop rows?** We only have ~7,000 customers. Dropping ~5% wastes data and introduces selection bias — missing tenure rows may lean toward newer customers.

**Why grouped median (by contract type) instead of global median?** A `Month-to-month` customer has a very different typical tenure from a `Two year` customer. The global median would over-estimate tenure for short-contract customers.

**Why median over mean?** `tenure` is right-skewed. Median is more robust to skew.

**Why add a `tenure_was_missing` flag?** Missingness may itself predict churn. The flag preserves this signal for the model.

In [ ]:
# Flag BEFORE filling — preserves the information that the value was missing
customers["tenure_was_missing"] = customers["tenure"].isna().astype(int)

# Fill with the median tenure within each contract type
customers["tenure"] = customers["tenure"].fillna(
    customers.groupby("contract")["tenure"].transform("median")
)

print(f"tenure nulls remaining:      {customers['tenure'].isna().sum()}")
print(f"tenure_was_missing flagged:  {customers['tenure_was_missing'].sum()} rows")
print()
print("Median tenure by contract (fill values used):")
print(customers.groupby("contract")["tenure"].median().to_string())

### 8c. `monthly_charges` — grouped median fill + missing flag

**Why group by `internet_service`?** Monthly charges are strongly driven by service tier — a `No` internet customer pays far less than a `Fiber Optic` customer. The global median would over- or under-estimate values depending on which tier the customer belongs to.

In [ ]:
customers["monthly_charges_was_missing"] = customers["monthly_charges"].isna().astype(int)

customers["monthly_charges"] = customers["monthly_charges"].fillna(
    customers.groupby("internet_service")["monthly_charges"].transform("median")
)

print(f"monthly_charges nulls remaining: {customers['monthly_charges'].isna().sum()}")
print()
print("Median monthly_charges by internet_service (fill values used):")
print(customers.groupby("internet_service")["monthly_charges"].median().to_string())

### 8d. `total_charges` — derive from existing columns

**Why derive instead of imputing with a median?** `total_charges` is not truly unknown — it can be calculated:
- Customers with `tenure == 0` are brand-new and have `total_charges = 0` by definition
- All other customers: `total_charges ≈ tenure × monthly_charges`

Filling with the column median would assign a long-tenure customer's total to a brand-new customer — logically wrong. When you can derive the correct answer from within the same row, always prefer derivation over statistical imputation.

In [ ]:
customers["total_charges"] = np.where(
    customers["total_charges"].isna(),
    np.where(
        customers["tenure"] == 0,
        0,
        customers["tenure"] * customers["monthly_charges"]
    ),
    customers["total_charges"]
)

print(f"total_charges nulls remaining: {customers['total_charges'].isna().sum()}")
print(f"total_charges range: {customers['total_charges'].min():.2f} – {customers['total_charges'].max():.2f}")

---
## 9. Validate `customers` is Fully Clean

Before touching any other dataset, we run a full validation pass. A clean dataframe should have zero nulls, correct dtypes, no duplicate `customer_id` values, and uppercase IDs ready for joining.

In [ ]:
print("=" * 55)
print("CUSTOMERS — CLEANING VALIDATION")
print("=" * 55)

print(f"\nShape: {customers.shape[0]:,} rows × {customers.shape[1]} columns")

nulls = customers.isna().sum()
remaining_nulls = nulls[nulls > 0]
if len(remaining_nulls) == 0:
    print("Nulls: none ✓")
else:
    print(f"WARNING — nulls still present:")
    print(remaining_nulls.to_string())

dupes = customers["customer_id"].duplicated().sum()
print(f"Duplicate customer_ids: {dupes} {'✓' if dupes == 0 else 'WARNING'}")

print(f"Churn rate: {customers['churn'].mean():.1%}  (should be ~26%)")

# Verify customer_id is still uppercase
sample_ids = customers["customer_id"].head(3).tolist()
ids_upper  = all(i == i.upper() for i in sample_ids)
print(f"customer_id uppercase: {'✓' if ids_upper else 'WARNING — IDs have been lowercased!'} — sample: {sample_ids}")

bad_names = [c for c in customers.columns if c != c.lower()]
print(f"Non-lowercase column names: {bad_names if bad_names else 'none ✓'}")

print(f"\nColumn dtypes:")
print(customers.dtypes.to_string())

---
## 10. Aggregate Supporting Datasets

**Why aggregate before merging?** `usage`, `tickets`, and `billing` each have multiple rows per customer. Merging them directly onto `customers` would give one customer with 10 tickets 10 rows in the final table — inflating row counts, duplicating profile data, and completely breaking any model trained on it.

**The rule:** reduce every supporting table to exactly one row per `customer_id` before merging.

### 10a. Aggregate `usage`

`days_since_last_login` is a recency feature — the higher the value, the more disengaged the customer. We compute it from the date difference, then drop the raw date column since models need numbers, not datetime objects.

In [ ]:
# Reference date: one day after the last login in the dataset
REFERENCE_DATE = usage["login_date"].max() + pd.Timedelta(days=1)
print(f"Reference date for recency: {REFERENCE_DATE.date()}")

usage_agg = usage.groupby("customer_id").agg(
    total_logins      = ("logins",           "sum"),
    avg_session_mins  = ("session_duration", "mean"),
    last_login_date   = ("login_date",        "max"),
    login_days_active = ("login_date",        "nunique")
).reset_index()

# Compute days since last login as an integer
usage_agg["days_since_last_login"] = (
    REFERENCE_DATE - usage_agg["last_login_date"]
).dt.days.astype(int)

# Drop the raw date — it is now encoded as an integer
usage_agg = usage_agg.drop(columns=["last_login_date"])

print(f"usage_agg shape: {usage_agg.shape}")
print(f"Unique customers covered: {usage_agg['customer_id'].nunique():,}")
print(f"days_since_last_login range: {usage_agg['days_since_last_login'].min()} – {usage_agg['days_since_last_login'].max()}")
usage_agg.head()

### 10b. Aggregate `tickets`

We break out billing tickets as a separate feature because billing disputes are the strongest ticket-type predictor of churn in telco research. A customer with 3 billing tickets is at much higher risk than one with 3 general enquiry tickets. `billing_ticket_ratio` normalises for overall volume — a customer with 1 billing ticket out of 1 total is different from 1 out of 10.

In [ ]:
tickets_agg = tickets.groupby("customer_id").agg(
    total_tickets        = ("issue_type",           "count"),
    avg_resolution_hours = ("resolution_time_hours", "mean"),
    max_resolution_hours = ("resolution_time_hours", "max"),
    billing_tickets      = ("issue_type",            lambda x: (x == "Billing").sum()),
    technical_tickets    = ("issue_type",            lambda x: (x == "Technical").sum())
).reset_index()

tickets_agg["billing_ticket_ratio"] = (
    tickets_agg["billing_tickets"] / tickets_agg["total_tickets"]
).round(3)

print(f"tickets_agg shape: {tickets_agg.shape}")
print(f"Unique customers covered: {tickets_agg['customer_id'].nunique():,}")
tickets_agg.head()

### 10c. Aggregate `billing`

We keep both `avg_payment_gap` and `max_payment_gap` — average captures general payment behaviour, maximum captures the worst single shortfall. A customer who usually pays on time but had one very large gap is still a churn risk that the average alone would miss.

In [ ]:
billing_agg = billing.groupby("customer_id").agg(
    total_bills         = ("amount_due",   "count"),
    total_late_payments = ("late_payment", "sum"),
    late_payment_rate   = ("late_payment", "mean"),
    avg_payment_gap     = ("payment_gap",  "mean"),
    max_payment_gap     = ("payment_gap",  "max"),
    avg_amount_due      = ("amount_due",   "mean")
).reset_index()

for col in ["late_payment_rate", "avg_payment_gap", "max_payment_gap", "avg_amount_due"]:
    billing_agg[col] = billing_agg[col].round(2)

print(f"billing_agg shape: {billing_agg.shape}")
print(f"Unique customers covered: {billing_agg['customer_id'].nunique():,}")
billing_agg.head()

### 10d. Final ID overlap check before merging

We confirm there is real overlap between all aggregated tables and `customers` before running the merge. If any overlap is 0 here, something is wrong with the ID standardisation and we must stop — a merge with 0 overlap silently produces all-NaN columns that then get filled with 0, making the data look clean while actually being useless.

In [ ]:
cust_ids = set(customers["customer_id"])

overlaps = {
    "usage_agg":   len(cust_ids & set(usage_agg["customer_id"])),
    "tickets_agg": len(cust_ids & set(tickets_agg["customer_id"])),
    "billing_agg": len(cust_ids & set(billing_agg["customer_id"]))
}

print("ID overlap before merge:")
all_ok = True
for name, n in overlaps.items():
    status = "✓" if n > 0 else "STOP — 0 overlap means the join will produce all NaN"
    print(f"  customers ∩ {name:<14}: {n:>5,}  {status}")
    if n == 0:
        all_ok = False

if not all_ok:
    raise ValueError("ID overlap is 0 for one or more tables. Check that customer_id is uppercase in all frames.")
else:
    print("\nAll overlaps > 0 — safe to merge ✓")

---
## 11. Merge All Tables into the Master DataFrame

**Why LEFT JOIN and not INNER JOIN?** An inner join drops every customer who has no usage records, tickets, or billing records. Since our simulated tables only cover a subset of customers, an inner join would silently discard thousands of rows — including churners the model needs. A left join keeps all customers and represents absent activity as NaN, which we then replace with 0.

**Why fill behavioural NaNs with 0 and not the column median?** For these columns, NaN means "this customer had zero activity", not "we don't know their activity". A customer with no ticket records has `total_tickets = 0`. Filling with the median would falsely imply average activity.

In [ ]:
df = customers.copy()
print(f"Base (customers):    {df.shape[0]:,} rows")

df = df.merge(usage_agg,   on="customer_id", how="left")
print(f"After + usage_agg:   {df.shape[0]:,} rows  (must not change)")

df = df.merge(tickets_agg, on="customer_id", how="left")
print(f"After + tickets_agg: {df.shape[0]:,} rows  (must not change)")

df = df.merge(billing_agg, on="customer_id", how="left")
print(f"After + billing_agg: {df.shape[0]:,} rows  (must not change)")

print(f"\nFinal shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Quick sanity check — if total_logins has non-zero values the join worked
pct_with_usage = (df["total_logins"] > 0).mean()
print(f"Customers with usage data:   {pct_with_usage:.1%}  (should be > 0%)")
pct_with_tickets = (df["total_tickets"] > 0).mean()
print(f"Customers with ticket data:  {pct_with_tickets:.1%}  (should be > 0%)")

### 11b. Fill behavioural NaNs and handle `days_since_last_login`

In [ ]:
behavioural_cols = [
    # from usage_agg
    "total_logins", "avg_session_mins", "login_days_active", "days_since_last_login",
    # from tickets_agg
    "total_tickets", "avg_resolution_hours", "max_resolution_hours",
    "billing_tickets", "technical_tickets", "billing_ticket_ratio",
    # from billing_agg
    "total_bills", "total_late_payments", "late_payment_rate",
    "avg_payment_gap", "max_payment_gap", "avg_amount_due"
]

present     = [c for c in behavioural_cols if c in df.columns]
missing_col = [c for c in behavioural_cols if c not in df.columns]
if missing_col:
    print(f"WARNING — expected columns not in df: {missing_col}")

# Fill all behavioural NaNs with 0 first
df[present] = df[present].fillna(0)

# Special case: days_since_last_login = 0 means 'logged in today'
# but for customers with NO usage records it should mean 'maximum disengagement'.
# We set it to the maximum observed value among customers who DID have login data.
if "days_since_last_login" in df.columns:
    max_days = df[df["total_logins"] > 0]["days_since_last_login"].max()
    df.loc[df["total_logins"] == 0, "days_since_last_login"] = int(max_days)
    print(f"days_since_last_login: non-users set to max observed value = {int(max_days)} days")

# Confirm
print(f"\nNon-zero value counts for key behavioural columns:")
for col in ["total_logins", "avg_session_mins", "total_tickets",
            "late_payment_rate", "avg_payment_gap", "days_since_last_login"]:
    if col in df.columns:
        nz   = (df[col] != 0).sum()
        nulls = df[col].isna().sum()
        print(f"  {col:<28}  non-zero={nz:>5,}  nulls={nulls}")

---
## 12. Final Validation of the Master Table

This is the last gate before saving. Nothing that fails here should be written to `processed/`.

In [ ]:
print("=" * 55)
print("MASTER TABLE — FINAL VALIDATION")
print("=" * 55)

print(f"\nShape: {df.shape[0]:,} rows × {df.shape[1]} columns")

expected_rows = customers.shape[0]
row_ok = df.shape[0] == expected_rows
print(f"Row count = {expected_rows:,} (matches clean customers): {'✓' if row_ok else 'WARNING'}")

dupes = df["customer_id"].duplicated().sum()
print(f"Duplicate customer_ids: {dupes} {'✓' if dupes == 0 else 'WARNING'}")

print(f"Churn rate: {df['churn'].mean():.1%}  (should be ~26%)")

nulls = df.isna().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("Nulls across all columns: none ✓")
else:
    print(f"WARNING — nulls remain:")
    print(nulls.to_string())

print(f"\nKey column ranges:")
range_checks = [
    ("tenure",              0,   72),
    ("monthly_charges",     0,  200),
    ("total_charges",       0, 9000),
    ("total_logins",        0, 9999),
    ("late_payment_rate",   0,    1),
    ("days_since_last_login", 1, 999),
]
for col, lo, hi in range_checks:
    if col in df.columns:
        mn, mx = df[col].min(), df[col].max()
        flag = "✓" if lo <= mn and mx <= hi else "⚠ outside expected range"
        print(f"  {col:<26}  min={mn:.1f}  max={mx:.1f}  {flag}")

print(f"\nAll {df.shape[1]} columns:")
print(df.columns.tolist())

---
## 13. Preview

In [ ]:
df.describe(include="all").T

In [ ]:
df.head()

---
## 14. Save to `data/processed/`

**Why `processed/` and not `raw/`?** `raw/` holds the original untouched files — the ground truth we can always restart from. The master table is a derived artefact produced by cleaning decisions that could be made differently. Keeping it separate means: change a cleaning decision in this notebook, re-run, and only `processed/` is updated — `raw/` is never touched.

In [ ]:
output_path = f"{PROCESSED_PATH}master_clean.csv"
df.to_csv(output_path, index=False)

size_kb = os.path.getsize(output_path) / 1024
print(f"Saved:  {output_path}")
print(f"Size:   {size_kb:.1f} KB")
print(f"Shape:  {df.shape[0]:,} rows × {df.shape[1]} columns")

---
## 15. Phase 2 Summary

In [ ]:
print("PHASE 2 COMPLETE — CLEANING & MERGING SUMMARY")
print("=" * 55)
print()
print("Critical fix applied:")
print("  customer_id standardised to uppercase across ALL tables FIRST")
print("  (prevents join failure caused by str.title() lowercasing IDs)")
print()
print("Cleaning decisions:")
print("  1. Duplicates       → drop_duplicates() full rows, then by customer_id")
print("  2. Column names     → regex snake_case conversion")
print("  3. total_charges    → pd.to_numeric(errors='coerce')")
print("  4. Category casing  → str.strip().str.title() EXCLUDING customer_id")
print("  5. tenure           → grouped median by contract + missing flag")
print("  6. monthly_charges  → grouped median by internet_service + missing flag")
print("  7. total_charges    → derived: 0 if tenure==0, else tenure×monthly")
print()
print("Aggregation:")
print(f"  usage_agg   → {usage_agg.shape[1]-1} features per customer")
print(f"  tickets_agg → {tickets_agg.shape[1]-1} features per customer")
print(f"  billing_agg → {billing_agg.shape[1]-1} features per customer")
print()
print("Merge: LEFT JOIN — preserves all customers")
print("Behavioural NaNs: filled with 0 (absence = zero activity)")
print("days_since_last_login: non-users set to max observed value")
print()
print(f"Output: master_clean.csv")
print(f"  {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  Churn rate: {df['churn'].mean():.1%}")
print()
print("Next: 03_eda_feature_engineering.ipynb")